In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
inventory_levels = pd.read_csv('INVENTORY_LEVELS-data.csv')
product_forecast = pd.read_csv('PRODUCT_FORECAST-data.csv')
sales_target = pd.read_csv('SALES_TARGET-data.csv', encoding='cp1252')

crm = sqlite3.connect('CRM-data.sqlite')
go_sales = sqlite3.connect('GO_SALES-data.sqlite')
go_staff = sqlite3.connect('GO_STAFF-data.sqlite')


In [3]:
age_group = pd.read_sql("SELECT * FROM age_group;",crm)
customer_contact = pd.read_sql("SELECT * FROM customer_contact;",crm)
customer_segment = pd.read_sql("SELECT * FROM customer_segment;",crm)
customer_type = pd.read_sql("SELECT * FROM customer_type;",crm)
sales_territory = pd.read_sql("SELECT * FROM sales_territory;",crm)
customer_store = pd.read_sql("SELECT * FROM customer_store;",crm)
crm_country = pd.read_sql("SELECT * FROM crm_country;",crm)
customer = pd.read_sql("SELECT * FROM customer;",crm)
sales_demographic = pd.read_sql("SELECT * FROM sales_demographic;",crm)
customer_headquarters = pd.read_sql("SELECT * FROM customer_headquarters;",crm)

order_method = pd.read_sql("SELECT * FROM order_method;",go_sales)
product_line = pd.read_sql("SELECT * FROM product_line;",go_sales)
retailer_site = pd.read_sql("SELECT * FROM retailer_site;",go_sales)
return_reason = pd.read_sql("SELECT * FROM return_reason;",go_sales)
returned_item = pd.read_sql("SELECT * FROM returned_item;",go_sales)
order_details = pd.read_sql("SELECT * FROM order_details;",go_sales)
order_header = pd.read_sql("SELECT * FROM order_header;",go_sales)
product_type = pd.read_sql("SELECT * FROM product_type;",go_sales)
product = pd.read_sql("SELECT * FROM product;",go_sales)
sales_staff = pd.read_sql("SELECT * FROM sales_staff;",go_sales)
country = pd.read_sql("SELECT * FROM country;",go_sales)
sales_branch = pd.read_sql("SELECT * FROM sales_branch;",go_sales)

satisfaction = pd.read_sql("SELECT * FROM satisfaction;",go_staff)
training = pd.read_sql("SELECT * FROM training;",go_staff)
course = pd.read_sql("SELECT * FROM course;",go_staff)
sales_office = pd.read_sql("SELECT * FROM sales_office;",go_staff)
satisfaction_type = pd.read_sql("SELECT * FROM satisfaction_type;",go_staff)
sales_representative = pd.read_sql("SELECT * FROM sales_representative;",go_staff)

In [4]:
# ik maak er een lijst van om in 1 overzicht te kunnen zien in welke databases een lege colom zit

lijstje = [
    ("inventory_levels", inventory_levels),
    ("product_forecast", product_forecast),
    ("sales_target", sales_target),
    ("age_group", age_group),
    ("customer_contact", customer_contact),
    ("customer_segment", customer_segment),
    ("customer_type", customer_type),
    ("sales_territory", sales_territory),
    ("customer_store", customer_store),
    ("crm_country", crm_country),
    ("customer", customer),
    ("sales_demographic", sales_demographic),
    ("customer_headquarters", customer_headquarters),
    ("order_method", order_method),
    ("product_line", product_line),
    ("retailer_site", retailer_site),
    ("return_reason", return_reason),
    ("order_details", order_details),
    ("order_header", order_header),
    ("product_type", product_type),
    ("product", product),
    ("sales_staff", sales_staff),
    ("country", country),
    ("sales_branch", sales_branch),
    ("satisfaction", satisfaction),
    ("training", training),
    ("course", course),
    ("sales_office", sales_office),
    ("satisfaction_type", satisfaction_type),
    ("sales_representative", sales_representative)
]


for naam, df in lijstje:
    for col in df.columns:
        if df[col].isna().any():
            print(f"{naam} - {col}")

customer_contact - EXTENSION
customer_contact - FAX
customer_contact - E_MAIL
customer_store - ADDITION
customer_store - STATE
customer_store - ZIPCODE
customer - CUSTOMER_CODEMR
customer_headquarters - ADDRESS2
customer_headquarters - REGION
customer_headquarters - POSTAL_ZONE
customer_headquarters - PHONE
customer_headquarters - FAX
retailer_site - ADDRESS2
retailer_site - REGION
retailer_site - POSTAL_ZONE
sales_staff - EXTENSION
sales_branch - ADDRESS2
sales_branch - REGION
sales_office - ADDITION
sales_office - REGION
sales_representative - EXTENSION


In [5]:
sales_representative['EXTENSION'] = sales_representative['EXTENSION'].fillna(0)

sales_office['ADDITION'] = sales_office['ADDITION'].fillna('-')
sales_office['REGION'] = sales_office['REGION'].fillna(sales_office['REGION'].mode()[0])

sales_branch['ADDRESS2'] = sales_branch['ADDRESS2'].fillna(sales_branch['ADDRESS1'].mode()[0])
sales_branch['REGION'] = sales_branch['REGION'].fillna(sales_branch['REGION'].mode()[0])

sales_staff['EXTENSION'] = sales_staff['EXTENSION'].fillna(sales_staff['EXTENSION'].mode()[0])

smollist = 'REGION', 'POSTAL_ZONE'
retailer_site['ADDRESS2'] = retailer_site['ADDRESS2'].fillna(retailer_site['ADDRESS1'])
for i in smollist:
    retailer_site[i] = retailer_site[i].fillna(retailer_site[i].mode()[0])

smollist = 'REGION', 'POSTAL_ZONE', 'PHONE', 'FAX'
customer_headquarters['ADDRESS2'] = customer_headquarters['ADDRESS2'].fillna(customer_headquarters['ADDRESS1'])
for i in smollist:
    customer_headquarters[i] = customer_headquarters[i].fillna(customer_headquarters[i].mode()[0])

customer['CUSTOMER_CODEMR'] = customer['CUSTOMER_CODEMR'].fillna(customer['CUSTOMER_CODEMR'].mode()[0])

smollist = 'STATE', 'ZIPCODE'

customer_store['ADDITION'] = customer_store['ADDITION'].fillna('-')
customer_store['STATE']
for i in smollist:
    customer_store[i] = customer_store[i].fillna(customer_store[i].mode()[0])

smollist = 'FAX', 'E_MAIL'
customer_contact['EXTENSION'] = customer_contact['EXTENSION'].fillna('0')
for i in smollist:
    customer_contact[i] = customer_contact[i].fillna(customer_contact[i].mode()[0])

product_forecast['PRODUCT_FORECAST_CODE'] = product_forecast.index
sales_target['SALES_TARGET_CODE'] = sales_target.index
inventory_levels['INVENTORY_LEVELS_CODE'] = inventory_levels.index
satisfaction['SATISFACTION_CODE'] = satisfaction.index
training['TRAINING_CODE'] = training.index

In [6]:
sales_representative = sales_representative.drop(columns='EXTENSION')

DIM_Sales_Staff = pd.merge(
    sales_representative,
    sales_staff,
    how='outer',
    on=['FIRST_NAME', 'LAST_NAME', 'POSITION_EN',
       'WORK_PHONE', 'FAX', 'EMAIL', 'DATE_HIRED']
)

DIM_Sales_Staff.insert(0, 'SALES_STAFF_CODE', DIM_Sales_Staff.pop('SALES_STAFF_CODE'))
DIM_Sales_Staff = DIM_Sales_Staff.drop(columns='SALES_REPRESENTATIVE_CODE')


In [7]:
crm_country = crm_country.rename(columns={'COUNTRY_EN':'COUNTRY'})

DIM_Country = pd.merge(
    country,
    crm_country,
    how='outer',
    on=['COUNTRY_CODE', 'COUNTRY']
)

In [8]:
sales_office = sales_office.rename(columns={'ZIPCODE':'POSTAL_ZONE',
                                            'COUNTRY': 'COUNTRY_CODE'})

DIM_Sales_Branch = pd.merge(
    sales_office,
    sales_branch,
    how='outer',
    on=['CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY_CODE']
)

DIM_Sales_Branch = DIM_Sales_Branch.drop(columns=['SALES_OFFICE_CODE', 'STREET'])

In [9]:
customer_store = customer_store.rename(columns={'ZIPCODE':'POSTAL_ZONE',
                                                'STATE': 'REGION'})

DIM_Retailer_Site = pd.merge(
    retailer_site,
    customer_store,
    how='outer',
    on=['CITY', 'REGION', 'POSTAL_ZONE', 'COUNTRY_CODE', 'ACTIVE_INDICATOR']
)

DIM_Retailer_Site = DIM_Retailer_Site.drop(columns=['CUSTOMER_SITE_CODE', 'STREET'])

In [10]:
return_reason

,RETURN_REASON_CODE,RETURN_DESCRIPTION_EN
0,1,Defective product
1,2,Incomplete product
2,3,Wrong product ordered
3,4,Wrong product shipped
4,5,Unsatisfactory product


In [11]:
returned_item = returned_item.merge(return_reason,on='RETURN_REASON_CODE',how='left')

In [12]:
#checklist = [inventory_levels, product_forecast, sales_target, returned_item, order_header, customer, customer_segment, age_group, sales_demographic, customer_headquarters, DIM_Country, DIM_Retailer_Site, DIM_Sales_Branch, Date, Month]
#anders = [product, order_details, DIM_Sales_Staff]


In [13]:
Month = pd.DataFrame()
Date = pd.DataFrame()

In [14]:
returned_item['RETURN_DATE'] = pd.to_datetime(returned_item['RETURN_DATE'], format="%d-%b-%Y %I:%M:%S %p")
product['INTRODUCTION_DATE'] = pd.to_datetime(product['INTRODUCTION_DATE'], format="%d-%b-%Y %I:%M:%S %p")
order_header['ORDER_DATE'] = pd.to_datetime(order_header['ORDER_DATE'], format="%d-%b-%Y %I:%M:%S %p")
DIM_Sales_Staff['DATE_HIRED'] = pd.to_datetime(DIM_Sales_Staff['DATE_HIRED'], format="%d-%b-%Y %I:%M:%S %p")

In [15]:
Month = pd.concat([
    pd.DataFrame({
        'YEAR': inventory_levels['INVENTORY_YEAR'],
        'MONTHNR': inventory_levels['INVENTORY_MONTH']
    }),
    
    pd.DataFrame({
        'YEAR': product_forecast['YEAR'],
        'MONTHNR': product_forecast['MONTH']
    })
]).drop_duplicates().sort_values(['YEAR', 'MONTHNR']).reset_index(drop=True)

Month['MONTH'] = Month['MONTHNR'].map({1 : 'Jan', 2 : 'Feb', 3 : 'Mar', 4 : 'Apr', 5:'Mei', 6 : 'Jun', 7 : 'Jul', 8 : 'Aug', 9 : 'Sep', 10 : 'Oct', 11 : 'Nov', 12 : 'Dec'})
Month['QUARTER'] = Month['MONTHNR'].map({ 2 : 1, 3 : 1, 4 : 2, 5:2, 6 : 2, 7 : 3, 8 : 3, 9 : 3, 10 : 4, 11 : 4, 12 : 4})
Month['MONTH_ID'] = Month.index

In [16]:
Date['DATE'] = pd.concat([
    returned_item['RETURN_DATE'],
    product['INTRODUCTION_DATE'],
    order_header['ORDER_DATE'],
    DIM_Sales_Staff['DATE_HIRED']
]).drop_duplicates().sort_values().reset_index(drop=True)

Date['DAY'] = pd.to_datetime(Date['DATE']).dt.day

Date['YEAR'] = Date['DATE'].dt.year
Date['MONTHNR'] = Date['DATE'].dt.month

Date = Date.merge(
    Month[['YEAR', 'MONTHNR', 'MONTH_ID']],
    on=['YEAR', 'MONTHNR'],
    how='left'
)

Date['DATE'] = Date['DATE'].astype(str)

In [17]:
DIM_Sales_Branch = DIM_Sales_Branch.rename(columns={'POSTAL_ZONE': 'POSTAL_CODE', 'COUNTRY_CODE':'COUNTRY'})
sales_office = sales_office.rename(columns={'POSTAL_CODE': 'ZIPCODE'})
returned_item = returned_item.rename(columns={'RETURN_DESCRIPTION_EN': 'RETURN_REASON_EN', 'ORDER_DETAIL_CODE':'ORDER_DETAILS'})
order_details = order_details.rename(columns={'PRODUCT_NUMBER': 'PRODUCT', 'ORDER_NUMBER': 'ORDER_HEADER'})


In [18]:
order_details['UNIT_PRICE'] = order_details['UNIT_PRICE'].astype('double')
order_details['UNIT_COST'] = order_details['UNIT_COST'].astype('double')

order_details['UNIT_REVENUE'] = (
    order_details['UNIT_PRICE'] - order_details['UNIT_COST']
)

In [19]:
export_base = sqlite3.connect('dwdb.db')
cur = export_base.cursor()

cur.execute("PRAGMA foreign_keys = OFF;")

# 1. geen FK
root = [
    ("DIM_Age_Group", age_group, ["AGE_GROUP_CODE", "UPPER_AGE", "LOWER_AGE"]),
    ("DIM_Customer_Segment", customer_segment, ["SEGMENT_CODE", "LANGUAGE", "SEGMENT_NAME", "SEGMENT_DESCRIPTION"]),
    ("DIM_Month", Month, ["MONTH_ID", "MONTHNR", "MONTH", "QUARTER", "YEAR"]),
]

# 2. 2de laag
mid = [
    ("DIM_Country", DIM_Country, ["COUNTRY_CODE", "COUNTRY", "LANGUAGE", "CURRENCY_NAME", "FLAG_IMAGE", "TERRITORY_NAME_EN"]),

    ("DIM_Date", Date, ["DATE", "DAY", "MONTH_ID"]),

    ("DIM_Sales_Branch", DIM_Sales_Branch, [
        "SALES_BRANCH_CODE", "ADDRESS1", "ADDRESS2", "CITY", "REGION", "POSTAL_CODE", "COUNTRY"
    ]),

    ("DIM_Sales_Office", sales_office, [
        "SALES_OFFICE_CODE", "STREET", "ADDITION", "CITY", "REGION", "ZIPCODE", 'COUNTRY_CODE'
    ]),

    ("DIM_Product", product, [
        "PRODUCT_NUMBER", "REGION", "INTRODUCTION_DATE", "PRODUCTION_CODE",
        "MARGIN", "PRODUCT_IMAGE", "LANGUAGE", "PRODUCT_NAME", "PRODUCT_TYPE_EN", "PRODUCT_LINE_EN"
    ]),
]

third = [
    ("DIM_Customer_Headquarters", customer_headquarters, [
        "CUSTOMER_CODE_HQ", "CUSTOMER_NAME", "ADRESS1", "ADRESS2",
        "CITY", "REGION", "POSTAL_ZONE", "PHONE", "FAX", "FLAG_IMAGE",
        "COUNTRY", "SEGMENT_CODE", "CUSTOMER_SEGMENT"
    ]),

    ("DIM_Sales_Staff", DIM_Sales_Staff, [
        "SALES_STAFF_CODE", "FIRST_NAME", "LAST_NAME", "POSITION_EN",
        "WORK_PHONE", "EXTENSION", "FAX", "EMAIL", "DATE_HIRED",
        "TRAINING_COURSE", "SATISFACTION_YEAR", "SATISFACTION_DESCRIPTION",
        "SALES_BRANCH", "MANAGER"
    ]),

    ("DIM_Sales_Representative", sales_representative, [
        "SALES_REPRESENTATIVE_CODE", "FIRST_NAME", "LAST_NAME", "POSITION_EN",
        "WORK_PHONE", "EXTENSION", "FAX", "EMAIL", "DATE_HIRED",
        "SALES_OFFICE", "MANAGER"
    ]),

    ("DIM_Customer", customer, [
        "CUSTOMER_CODE", "COMPANY_NAME", "CUSTOMER_TYPE", "CUSTOMER_HEADQUARTERS"
    ]),

    ("DIM_Retailer_Site", DIM_Retailer_Site, [
        "RETAILER_SITE_CODE", "RETAILER_CODE", "ADDRESS1", "ADDRESS2",
        "CITY", "REGION", "POSTAL_ZONE", "ACTIVE_INDICATOR", "CUSTOMER", "COUNTRY"
    ]),

    ("DIM_Customer_Store", customer_store, [
        "CUSTOMER_STORE_CODE", "STREET", "ADDITION", "CITY", "STATE",
        "ZIPCODE", "ACTIVE_INDICATOR", "CUSTOMER", "CRM_COUNTRY"
    ]),
]

fact = [
    ("FACT_Sales_Demographic", sales_demographic, [
        "DEMOGRAPHIC_CODE", "SALES_PERCENT", "CUSTOMER_HEADQUARTERS", "AGE_GROUP"
    ]),

    ("FACT_Order_Header", order_header, [
        "ORDER_NUMBER", "RETAILER_NAME", "RETAILER_CONTACT_CODE",
        "ORDER_DATE", "ORDER_METHOD_EN", "RETAILER_SITE", "SALES_STAFF", "SALES_BRANCH"
    ]),

    ("FACT_Order_Details", order_details, [
        "ORDER_DETAIL_CODE", "QUANTITY", "UNIT_COST", "UNIT_PRICE",
        "UNIT_REVENUE", "UNIT_SALE_PRICE", "ORDER_HEADER", "PRODUCT"
    ]),

    ("FACT_Returned_Item", returned_item, [
        "RETURN_CODE", "RETURN_DATE", "RETURN_QUANTITY", "RETURN_REASON_EN", "ORDER_DETAILS"
    ]),

    ("DIM_Inventory_Levels", inventory_levels, [
        "INVENTORY_LEVELS_CODE", "INVENTORY_YEAR", "INVENTORY_MONTH", "INVENTORY_COUNT", "PRODUCT_NUMBER"
    ]),

    ("DIM_Product_Forecast", product_forecast, [
        "PRODUCT_FORECAST_CODE", "YEAR", "MONTH", "EXPECTED_VOLUME", "PRODUCT_NUMBER"
    ]),

    ("FACT_Sales_Target", sales_target, [
        "SALES_TARGET_CODE", "SALES_YEAR", "SALES_PERIOD",
        "RETAILER_NAME", "RETAILER_CODE", "EXPECTED_VOLUME", "SALES_TARGET",
        "SALES_STAFF_CODE", "PRODUCT_NUMBER"
    ]),
]

all_tables = root + mid + third + fact

for table_name, df, cols in all_tables:

    missing = set(cols) - set(df.columns)
    if missing:
        print(f"❌ {table_name} mist: {missing}")
        continue

    df_clean = df[cols]

    query = f"""
    INSERT OR REPLACE INTO {table_name}
    ({",".join(cols)})
    VALUES ({",".join(["?"] * len(cols))})
    """

    try:
        cur.executemany(query, df_clean.values.tolist())
        print(f"✔ {table_name}")
    except Exception as e:
        print(f"❌ {table_name} ERROR: {e}")
        break

export_base.commit()
export_base.close()

✔ DIM_Age_Group
✔ DIM_Customer_Segment
✔ DIM_Month
❌ DIM_Country mist: {'TERRITORY_NAME_EN'}
✔ DIM_Date
✔ DIM_Sales_Branch
❌ DIM_Sales_Office mist: {'ZIPCODE'}
❌ DIM_Product mist: {'PRODUCT_TYPE_EN', 'PRODUCT_LINE_EN', 'REGION', 'PRODUCTION_CODE'}
❌ DIM_Customer_Headquarters mist: {'ADRESS1', 'COUNTRY', 'CUSTOMER_CODE_HQ', 'CUSTOMER_SEGMENT', 'ADRESS2', 'FLAG_IMAGE'}
❌ DIM_Sales_Staff mist: {'TRAINING_COURSE', 'SATISFACTION_YEAR', 'MANAGER', 'SALES_BRANCH', 'SATISFACTION_DESCRIPTION'}
❌ DIM_Sales_Representative mist: {'EXTENSION', 'SALES_OFFICE', 'MANAGER'}
❌ DIM_Customer mist: {'CUSTOMER_TYPE', 'CUSTOMER_HEADQUARTERS'}
❌ DIM_Retailer_Site mist: {'CUSTOMER', 'COUNTRY'}
❌ DIM_Customer_Store mist: {'ZIPCODE', 'CUSTOMER_STORE_CODE', 'CUSTOMER', 'STATE', 'CRM_COUNTRY'}
❌ FACT_Sales_Demographic mist: {'CUSTOMER_HEADQUARTERS', 'AGE_GROUP'}
❌ FACT_Order_Header mist: {'ORDER_METHOD_EN', 'RETAILER_SITE', 'SALES_STAFF', 'SALES_BRANCH'}
✔ FACT_Order_Details
❌ FACT_Returned_Item ERROR: Error bindi